# Gradient Boosting (Turkey, Pipeline B)

**Model**: `sklearn.ensemble.GradientBoostingRegressor` / **Target**: `ngdprsaxdctrq`  
**Variables**: Cat3 (22 vars + 3 COVID = 25 total)  
**Train**: 1995-Q2 to 2011-Q4 / **Val**: 2012-Q1 to 2017-Q4 / **Test**: 2018-Q1 to 2025-Q4 (32 quarters)  
**Scaling**: NO / **HP tuning**: YES — GridSearchCV(TimeSeriesSplit(5)) on train+val / **n_lags**: 6  
**Note**: Turkey keeps n_lags=6 (original). With 22 Cat3 vars: 22×7+3=157 features / 67 obs = 2.34 ratio.  
**10-model averaging** / **n_jobs=1** (Windows constraint)


In [1]:
import sys, os
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

sys.path.insert(0, os.path.join(os.path.pardir, 'turkey_data'))
from turkey_helpers import (
    gen_lagged_data, make_supervised_vintage_frame, flatten_data, mean_fill_dataset,
    get_features, load_data,
    PREDICTIONS_DIR, TARGET, COVID
)

SEED = 42; np.random.seed(SEED)
TRAIN_START = '1995-01-01'; TRAIN_END = '2011-12-31'; VAL_END = '2017-12-31'
TEST_START  = '2018-01-01'; TEST_END  = '2025-12-31'
N_LAGS = 6; N_MODELS = 10  # Turkey keeps original n_lags=6 (22 vars, manageable ratio)
VINTAGES = {'m1': -2, 'm2': -1, 'm3': 0, 'post1': 1, 'post2': 2}

print('GB (Turkey) — Cat3, n_lags={}, scaling=NO, 10-model avg'.format(N_LAGS))


GB (Turkey) — Cat3, n_lags=6, scaling=NO, 10-model avg


In [2]:
monthly, _, metadata = load_data()
features = get_features('cat3', with_covid=True)
print('Features: {} + lags = {} lagged features per quarter'.format(len(features), len(features)*(N_LAGS+1)))

# Phase A: HP tuning on train+val (1995-2017)
tune_data   = monthly[(monthly['date'] >= TRAIN_START) & (monthly['date'] <= VAL_END)].reset_index(drop=True)
tune_filled = mean_fill_dataset(tune_data, tune_data)
tune_flat   = flatten_data(tune_filled, TARGET, N_LAGS)
tune_flat   = tune_flat.loc[tune_flat.date.dt.month.isin([3,6,9,12]), :].dropna(axis=0, how='any').reset_index(drop=True)

feature_cols = [
    c for c in tune_flat.columns
    if c != 'date' and c != TARGET
    and any(c == f or c.startswith(f + '_') for f in features)
]
print('Effective feature columns: {}, training quarters: {}'.format(len(feature_cols), len(tune_flat)))

tscv      = TimeSeriesSplit(n_splits=5)
gb_base   = GradientBoostingRegressor(random_state=SEED)
gb_grid   = {'n_estimators': [100, 300], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1], 'subsample': [0.8, 1.0]}
gb_search = GridSearchCV(gb_base, gb_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=1)
gb_search.fit(tune_flat[feature_cols].values, tune_flat[TARGET].values)
best_params = gb_search.best_params_
print('Best GB params: {}'.format(best_params))


Features: 25 + lags = 175 lagged features per quarter


Effective feature columns: 175, training quarters: 90


Best GB params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}


In [3]:
# Phase B: Rolling test loop with 10-model averaging
test_quarters = monthly[
    (monthly['date'] >= TEST_START) &
    (monthly['date'] <= TEST_END) &
    monthly['date'].dt.month.isin([3,6,9,12])
]['date'].tolist()

predictions = {v: [] for v in VINTAGES}; actuals_list = []; failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actuals_list.append(monthly[monthly['date'] == pd_q][TARGET].values[0])

    for vn, offset in VINTAGES.items():
        vintage_date = pd_q + pd.DateOffset(months=offset)
        train_fl, tr_fl, _ = make_supervised_vintage_frame(
            metadata, monthly, TARGET, features, TRAIN_START, pd_q, vintage_date, N_LAGS
        )

        if len(train_fl) < 10 or len(tr_fl) != 1:
            predictions[vn].append(np.nan); continue

        try:
            models = []
            for k in range(N_MODELS):
                m = GradientBoostingRegressor(
                    n_estimators=best_params['n_estimators'],
                    max_depth=best_params['max_depth'],
                    learning_rate=best_params['learning_rate'],
                    subsample=best_params['subsample'],
                    random_state=SEED+k)
                m.fit(train_fl[feature_cols].values, train_fl[TARGET].values)
                models.append(m)

            preds  = [m.predict(tr_fl[feature_cols].values)[0] for m in models]
            predictions[vn].append(np.nanmean(preds))
        except Exception:
            predictions[vn].append(np.nan); failed += 1

    if (i+1) % 8 == 0:
        print('  {} / {}'.format(i+1, len(test_quarters)))

print('Done. {} failures.'.format(failed))


  8 / 32


  16 / 32


  24 / 32


  32 / 32
Done. 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    df = pd.DataFrame({'date': test_quarters, 'actual': actuals_list, 'prediction': predictions[vn]})
    path = os.path.join(PREDICTIONS_DIR, 'gb_{}.csv'.format(vn))
    df.to_csv(path, index=False)
    print('Saved {} ({} rows) pred=[{:.4f},{:.4f}]'.format(
        os.path.basename(path), len(df), df['prediction'].min(), df['prediction'].max()))


Saved gb_m1.csv (32 rows) pred=[-0.0119,0.0492]
Saved gb_m2.csv (32 rows) pred=[-0.0162,0.0390]
Saved gb_m3.csv (32 rows) pred=[-0.0266,0.0435]
Saved gb_post1.csv (32 rows) pred=[-0.0304,0.0431]
Saved gb_post2.csv (32 rows) pred=[-0.0370,0.0429]


In [5]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum()>0 else np.nan

def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m]-p[m])) if m.sum()>0 else np.nan

panels = {
    'Pre-crisis (2018-2019)': ('2018-01-01', '2019-12-31'),
    'COVID      (2020-2021)': ('2020-04-01', '2021-12-31'),
    'Post-COVID (2022-2025)': ('2022-01-01', '2025-12-31'),
    'Full test  (2018-2025)': ('2018-01-01', '2025-12-31'),
}
a = np.array(actuals_list); d = pd.to_datetime(test_quarters)
print('GB (Turkey) — Evaluation by Panel & Vintage')
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print('--- {} ---'.format(pn))
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print('  {}  RMSFE={:.6f}  MAE={:.6f}  N={}'.format(vn, rmse(a[m],pp[m]), mae(a[m],pp[m]), m.sum()))


GB (Turkey) — Evaluation by Panel & Vintage
--- Pre-crisis (2018-2019) ---
  m1  RMSFE=0.012968  MAE=0.009528  N=8
  m2  RMSFE=0.012881  MAE=0.010617  N=8
  m3  RMSFE=0.012018  MAE=0.010214  N=8
  post1  RMSFE=0.012887  MAE=0.010479  N=8
  post2  RMSFE=0.014214  MAE=0.012961  N=8
--- COVID      (2020-2021) ---
  m1  RMSFE=0.066261  MAE=0.043163  N=7
  m2  RMSFE=0.064173  MAE=0.040594  N=7
  m3  RMSFE=0.061398  MAE=0.041753  N=7
  post1  RMSFE=0.062078  MAE=0.042663  N=7
  post2  RMSFE=0.061754  MAE=0.043327  N=7
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.019043  MAE=0.016162  N=16
  m2  RMSFE=0.017668  MAE=0.015023  N=16
  m3  RMSFE=0.016326  MAE=0.012299  N=16
  post1  RMSFE=0.015629  MAE=0.012083  N=16
  post2  RMSFE=0.015480  MAE=0.012210  N=16
--- Full test  (2018-2025) ---
  m1  RMSFE=0.034518  MAE=0.020395  N=32
  m2  RMSFE=0.033261  MAE=0.019542  N=32
  m3  RMSFE=0.031628  MAE=0.018280  N=32
  post1  RMSFE=0.031838  MAE=0.018462  N=32
  post2  RMSFE=0.031744  MAE=0.019137  N=3